In [ ]:
import pandas as pd
from vali_tools.valitools import check_health
import os
import numpy as np
import pickle
import datetime
pd.set_option('display.max_columns', None)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
filename = os.listdir(r'.\feature')[-1]
df = pd.read_parquet(r'.\feature\{}'.format(filename))
df = df[df.survival_rate.notnull()]

In [ ]:
df

In [ ]:
check_health(df)

# Target 1: Survival Rate

In [ ]:
#traint test definition
X = df.drop(['cycle_id','weight','size','avg_weight_har','selling_price','total_biocunt','last_sampled_weight','generated_date','survival_rate'], axis=1)
y_1 = df['survival_rate']
y_2 = df['avg_weight_har']
y_3 = df['selling_price']
y_4 = df['weight']


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_1, test_size=0.2, random_state=42)

# Define preprocessing for numerical and categorical features
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

#
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Create a pipeline with preprocessing and model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))  # Change to AdaBoostRegressor() if needed
])

# Define hyperparameter grid for tuning
param_grid = {
    'regressor': [RandomForestRegressor(random_state=42)],
    'regressor__n_estimators': [200,400],
    'regressor__max_depth': [5,10, 20] if isinstance(pipeline.named_steps['regressor'], RandomForestRegressor) else [None]
}

# Use GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_squared_error')

# Fit the model
grid_search.fit(X_train, y_train)

# Evaluate the model
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error: {mse}')

# Best model
print(f'Best model: {grid_search.best_estimator_}')
# model path
version = "version_{}".format(datetime.datetime.now().strftime("%Y%m%d%H%M%S"))
pickle_file_path = 'model_repo\sr\model_sr_{}.pkl'.format(version)

# Export the best_model to a pickle file
with open(pickle_file_path, 'wb') as file:
    pickle.dump(grid_search, file)

In [ ]:
best_model = grid_search.best_estimator_
feature_names = numeric_features.tolist() + best_model.named_steps['preprocessor'].transformers_[1][1].named_steps['onehot'].get_feature_names_out(categorical_features).tolist()
feature_importances = best_model.named_steps['regressor'].feature_importances_
feature_importances_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

feature_importances_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.to_excel(r'Report\feature_importances_sr_model_randomforest.xlsx', index=False)

In [ ]:
#pickle load model
with open(r"model_repo\biomass\model_biomass_version_20240907233653.pkl", 'rb') as file:
    model = pickle.load(file)

#use model to predict
test_data = pd.read_excel(r"devs\X_test.xlsx")
y_pred = model.predict(X_test)
outcome = pd.DataFrame({'pridiction result': y_pred})
outcome

In [ ]:
#create function to predict based on loaded pickle model and store the result in a dataframe for streamlit purpose
def predict_model(model, test_data):
    y_pred = model.predict(test_data)
    outcome = pd.DataFrame({'pridiction result': y_pred})
    return outcome


# Target 2: average weight harverst in grams

In [ ]:
#traint test definition
X = df.drop(['cycle_id','weight','size','avg_weight_har','selling_price','total_biocunt','last_sampled_weight','generated_date','survival_rate'], axis=1)
y_1 = df['survival_rate']
y_2 = df['avg_weight_har']
y_3 = df['selling_price']
y_4 = df['weight']


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_2, test_size=0.2, random_state=42)

# Define preprocessing for numerical and categorical features
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

#
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Create a pipeline with preprocessing and model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))  # Change to AdaBoostRegressor() if needed
])

# Define hyperparameter grid for tuning
param_grid = {
    'regressor': [RandomForestRegressor(random_state=42)],
    'regressor__n_estimators': [200,400,500],
    'regressor__max_depth': [5,10, 20] if isinstance(pipeline.named_steps['regressor'], RandomForestRegressor) else [None]
}

# Use GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_squared_error')

# Fit the model
grid_search.fit(X_train, y_train)

# Evaluate the model
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error: {mse}')

# Best model
print(f'Best model: {grid_search.best_estimator_}')

# model path
version = "version_{}".format(datetime.datetime.now().strftime("%Y%m%d%H%M%S"))
pickle_file_path = 'model_repo\\avg_harvest\model_avghar_{}.pkl'.format(version)

# Export the best_model to a pickle file
with open(pickle_file_path, 'wb') as file:
    pickle.dump(grid_search, file)

In [ ]:
best_model = grid_search.best_estimator_
feature_names = numeric_features.tolist() + best_model.named_steps['preprocessor'].transformers_[1][1].named_steps['onehot'].get_feature_names_out(categorical_features).tolist()
feature_importances = best_model.named_steps['regressor'].feature_importances_
feature_importances_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

feature_importances_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.to_excel(r'Report\feature_importances_avghar_model_randomforest.xlsx', index=False)

# Target3: Selling Price

In [ ]:
#traint test definition
X = df.drop(['cycle_id','weight','size','avg_weight_har','selling_price','total_biocunt','last_sampled_weight','generated_date','survival_rate'], axis=1)
y_1 = df['survival_rate']
y_2 = df['avg_weight_har']
y_3 = df['selling_price']
y_4 = df['weight']


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_3, test_size=0.2, random_state=42)

# Define preprocessing for numerical and categorical features
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

#
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Create a pipeline with preprocessing and model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))  # Change to AdaBoostRegressor() if needed
])

# Define hyperparameter grid for tuning
param_grid = {
    'regressor': [RandomForestRegressor(random_state=42)],
    'regressor__n_estimators': [200,400,500],
    'regressor__max_depth': [5,10, 20] if isinstance(pipeline.named_steps['regressor'], RandomForestRegressor) else [None]
}

# Use GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_squared_error')

# Fit the model
grid_search.fit(X_train, y_train)

# Evaluate the model
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error: {mse}')

# Best model
print(f'Best model: {grid_search.best_estimator_}')

version = "version_{}".format(datetime.datetime.now().strftime("%Y%m%d%H%M%S"))
pickle_file_path = 'model_repo\\sellprice\model_sellprice_{}.pkl'.format(version)

# Export the best_model to a pickle file
with open(pickle_file_path, 'wb') as file:
    pickle.dump(grid_search, file)

In [ ]:
best_model = grid_search.best_estimator_
feature_names = numeric_features.tolist() + best_model.named_steps['preprocessor'].transformers_[1][1].named_steps['onehot'].get_feature_names_out(categorical_features).tolist()
feature_importances = best_model.named_steps['regressor'].feature_importances_
feature_importances_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

feature_importances_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.to_excel(r'Report\feature_importances_sellprice_model_randomforest.xlsx', index=False)

# Target 4: Biomass

In [ ]:
#traint test definition
X = df.drop(['cycle_id','weight','size','avg_weight_har','selling_price','total_biocunt','last_sampled_weight','generated_date','survival_rate'], axis=1)
y_1 = df['survival_rate']
y_2 = df['avg_weight_har']
y_3 = df['selling_price']
y_4 = df['weight']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_4, test_size=0.2, random_state=42)

# Define preprocessing for numerical and categorical features
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

#
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Create a pipeline with preprocessing and model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))  # Change to AdaBoostRegressor() if needed
])

# Define hyperparameter grid for tuning
param_grid = {
    'regressor': [RandomForestRegressor(random_state=42)],
    'regressor__n_estimators': [200,400,500],
    'regressor__max_depth': [5,10, 20] if isinstance(pipeline.named_steps['regressor'], RandomForestRegressor) else [None]
}

# Use GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_squared_error')

# Fit the model
grid_search.fit(X_train, y_train)

# Evaluate the model
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error: {mse}')

# Best model
print(f'Best model: {grid_search.best_estimator_}')

version = "version_{}".format(datetime.datetime.now().strftime("%Y%m%d%H%M%S"))
pickle_file_path = 'model_repo\\biomass\model_biomass_{}.pkl'.format(version)

# Export the best_model to a pickle file
with open(pickle_file_path, 'wb') as file:
    pickle.dump(grid_search, file)

In [ ]:
best_model = grid_search.best_estimator_
feature_names = numeric_features.tolist() + best_model.named_steps['preprocessor'].transformers_[1][1].named_steps['onehot'].get_feature_names_out(categorical_features).tolist()
feature_importances = best_model.named_steps['regressor'].feature_importances_
feature_importances_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

feature_importances_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.to_excel(r'Report\feature_importances_biomass_model_randomforest.xlsx', index=False)